# [FMA: A Dataset For Music Analysis](https://github.com/mdeff/fma)

Michaël Defferrard, Kirell Benzi, Pierre Vandergheynst, Xavier Bresson, EPFL LTS2.

## Baselines

* This notebook evaluates standard classifiers from scikit-learn on the provided features.
* Moreover, it evaluates Deep Learning models on both audio and spectrograms.

In [18]:

import time
import os

import IPython.display as ipd
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras.layers import Activation, Dense, Conv1D, Conv2D, MaxPooling1D, Flatten, Reshape

from sklearn.utils import shuffle
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder, LabelBinarizer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
#from sklearn.gaussian_process import GaussianProcessClassifier
#from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.multiclass import OneVsRestClassifier

import utils


In [19]:

# ── Intel XPU setup ──────────────────────────────────────────────────────────
# Importing ITEX registers the XPU device with TensorFlow so Keras ops
# are automatically dispatched to the Intel Arc GPU when available.
try:
    import intel_extension_for_tensorflow as itex
    print(f"ITEX {itex.__version__} loaded")
except ImportError as e:
    itex = None
    print(f"ITEX not available ({e}); falling back to CPU/CUDA")

def _try_device(device):
    try:
        with tf.device(device):
            tf.matmul(tf.ones([2, 2]), tf.ones([2, 2]))
        return True
    except Exception:
        return False

def pick_best_device():
    if tf.config.list_physical_devices("GPU") and _try_device("/GPU:0"):
        return "/GPU:0"
    if tf.config.list_physical_devices("XPU") and _try_device("/XPU:0"):
        return "/XPU:0"
    return "/CPU:0"

DEVICE = pick_best_device()
print(f"Using device: {DEVICE}")
print("Physical devices:", tf.config.list_physical_devices())


ITEX 2.15.0.3 loaded
Using device: /XPU:0
Physical devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:XPU:0', device_type='XPU')]


In [20]:
AUDIO_DIR = os.environ.get('AUDIO_DIR', "/mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_small")

tracks = utils.load('/mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_metadata/tracks.csv')
features = utils.load('/mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_metadata/features.csv')
echonest = utils.load('/mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_metadata/echonest.csv')

np.testing.assert_array_equal(features.index, tracks.index)
assert echonest.index.isin(tracks.index).all()

tracks.shape, features.shape, echonest.shape

((106574, 52), (106574, 518), (13129, 249))

## Subset

In [21]:
subset = tracks.index[tracks['set', 'subset'] <= 'small']

assert subset.isin(tracks.index).all()
assert subset.isin(features.index).all()

features_all = features.join(echonest, how='inner').sort_index(axis=1)
print('Not enough Echonest features: {}'.format(features_all.shape))

tracks = tracks.loc[subset]
features_all = features.loc[subset]

tracks.shape, features_all.shape

Not enough Echonest features: (13129, 767)


((8000, 52), (8000, 518))

In [22]:
train = tracks.index[tracks['set', 'split'] == 'training']
val = tracks.index[tracks['set', 'split'] == 'validation']
test = tracks.index[tracks['set', 'split'] == 'test']

print('{} training examples, {} validation examples, {} testing examples'.format(*map(len, [train, val, test])))

genres = list(LabelEncoder().fit(tracks['track', 'genre_top']).classes_)
#genres = list(tracks['track', 'genre_top'].unique())
print('Top genres ({}): {}'.format(len(genres), genres))
genres = list(MultiLabelBinarizer().fit(tracks['track', 'genres_all']).classes_)
print('All genres ({}): {}'.format(len(genres), genres))

6400 training examples, 800 validation examples, 800 testing examples
Top genres (8): ['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']
All genres (114): [1, 2, 6, 10, 12, 15, 16, 17, 18, 21, 22, 25, 26, 27, 30, 31, 32, 33, 36, 38, 41, 42, 45, 46, 47, 49, 53, 58, 64, 66, 70, 71, 76, 77, 79, 81, 83, 85, 86, 88, 89, 90, 92, 94, 98, 100, 101, 102, 103, 107, 109, 111, 113, 117, 118, 125, 130, 167, 171, 172, 174, 177, 180, 181, 182, 183, 184, 185, 186, 214, 224, 232, 236, 240, 247, 250, 267, 286, 296, 297, 314, 337, 359, 360, 361, 362, 400, 401, 404, 439, 440, 456, 468, 491, 495, 502, 504, 514, 524, 538, 539, 542, 580, 602, 619, 695, 741, 763, 808, 811, 1032, 1060, 1193, 1235]


## 1 Multiple classifiers and feature sets

Todo:
* Cross-validation for hyper-parameters.
* Dimensionality reduction?

### 1.1 Pre-processing

In [23]:
def pre_process(tracks, features, columns, multi_label=False, verbose=False):
    if not multi_label:
        # Assign an integer value to each genre.
        enc = LabelEncoder()
        labels = tracks['track', 'genre_top']
        #y = enc.fit_transform(tracks['track', 'genre_top'])
    else:
        # Create an indicator matrix.
        enc = MultiLabelBinarizer()
        labels = tracks['track', 'genres_all']
        #labels = tracks['track', 'genres']

    # Split in training, validation and testing sets.
    y_train = enc.fit_transform(labels[train])
    y_val = enc.transform(labels[val])
    y_test = enc.transform(labels[test])
    X_train = features.loc[train, columns].values
    X_val = features.loc[val, columns].values
    X_test = features.loc[test, columns].values
    
    X_train, y_train = shuffle(X_train, y_train, random_state=42)
    
    # Standardize features by removing the mean and scaling to unit variance.
    scaler = StandardScaler(copy=False)
    scaler.fit_transform(X_train)
    scaler.transform(X_val)
    scaler.transform(X_test)
    
    return y_train, y_val, y_test, X_train, X_val, X_test

### 1.2 Single genre

In [24]:
def test_classifiers_features(classifiers, feature_sets, multi_label=False):
    columns = list(classifiers.keys()).insert(0, 'dim')
    scores = pd.DataFrame(columns=columns, index=feature_sets.keys())
    times = pd.DataFrame(columns=classifiers.keys(), index=feature_sets.keys())
    for fset_name, fset in tqdm(feature_sets.items(), desc='features'):
        y_train, y_val, y_test, X_train, X_val, X_test = pre_process(tracks, features_all, fset, multi_label)
        scores.loc[fset_name, 'dim'] = X_train.shape[1]
        for clf_name, clf in classifiers.items():  # tqdm(classifiers.items(), desc='classifiers', leave=False):
            t = time.process_time()
            clf.fit(X_train, y_train)
            score = clf.score(X_test, y_test)
            scores.loc[fset_name, clf_name] = score
            times.loc[fset_name, clf_name] = time.process_time() - t
    return scores, times

def format_scores(scores):
    def highlight(s):
        is_max = s == max(s[1:])
        return ['background-color: yellow' if v else '' for v in is_max]
    scores = scores.style.apply(highlight, axis=1)
    return scores.format('{:.2%}', subset=pd.IndexSlice[:, scores.columns[1]:])

In [25]:

classifiers = {
    'LR': LogisticRegression(),
    'kNN': KNeighborsClassifier(n_neighbors=200),
    'SVCrbf': SVC(kernel='rbf'),
    'SVCpoly1': SVC(kernel='poly', degree=1),
    'linSVC1': SVC(kernel="linear"),
    'linSVC2': LinearSVC(),
    #GaussianProcessClassifier(1.0 * RBF(1.0), warm_start=True),
    'DT': DecisionTreeClassifier(max_depth=5),
    'RF': RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1),
    'AdaBoost': AdaBoostClassifier(n_estimators=10),
    'MLP1': MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000),
    'MLP2': MLPClassifier(hidden_layer_sizes=(200, 50), max_iter=2000),
    'NB': GaussianNB(),
    'QDA': QuadraticDiscriminantAnalysis(reg_param=0.1),
}

feature_sets = {
#    'echonest_audio': ('echonest', 'audio_features'),
#    'echonest_social': ('echonest', 'social_features'),
#    'echonest_temporal': ('echonest', 'temporal_features'),
#    'echonest_audio/social': ('echonest', ('audio_features', 'social_features')),
#    'echonest_all': ('echonest', ('audio_features', 'social_features', 'temporal_features')),
}
for name in features.columns.levels[0]:
    feature_sets[name] = name
feature_sets.update({
    'mfcc/contrast': ['mfcc', 'spectral_contrast'],
    'mfcc/contrast/chroma': ['mfcc', 'spectral_contrast', 'chroma_cens'],
    'mfcc/contrast/centroid': ['mfcc', 'spectral_contrast', 'spectral_centroid'],
    'mfcc/contrast/chroma/centroid': ['mfcc', 'spectral_contrast', 'chroma_cens', 'spectral_centroid'],
    'mfcc/contrast/chroma/centroid/tonnetz': ['mfcc', 'spectral_contrast', 'chroma_cens', 'spectral_centroid', 'tonnetz'],
    'mfcc/contrast/chroma/centroid/zcr': ['mfcc', 'spectral_contrast', 'chroma_cens', 'spectral_centroid', 'zcr'],
    'all_non-echonest': list(features.columns.levels[0])
})

# scores, times = test_classifiers_features(classifiers, feature_sets)

# ipd.display(format_scores(scores))
# ipd.display(times.style.format('{:.4f}'))

### 1.3 Multiple genres

Todo:
* Ignore rare genres? Count them higher up in the genre tree? On the other hand it's not much tracks.

In [26]:
classifiers = {
    #LogisticRegression(),
    'LR': OneVsRestClassifier(LogisticRegression()),
    'SVC': OneVsRestClassifier(SVC()),
    'MLP': MLPClassifier(max_iter=700),
}

feature_sets = {
#    'echonest_audio': ('echonest', 'audio_features'),
#    'echonest_temporal': ('echonest', 'temporal_features'),
    'mfcc': 'mfcc',
    'mfcc/contrast/chroma/centroid/tonnetz': ['mfcc', 'spectral_contrast', 'chroma_cens', 'spectral_centroid', 'tonnetz'],
    'mfcc/contrast/chroma/centroid/zcr': ['mfcc', 'spectral_contrast', 'chroma_cens', 'spectral_centroid', 'zcr'],
}

# scores, times = test_classifiers_features(classifiers, feature_sets, multi_label=True)

# ipd.display(format_scores(scores))
# ipd.display(times.style.format('{:.4f}'))

## 2 Deep learning on raw audio

Other architectures:
* [Learning Features of Music from Scratch (MusicNet)](https://arxiv.org/abs/1611.09827), John Thickstun, Zaid Harchaoui, Sham Kakade.

In [27]:
labels_onehot = LabelBinarizer().fit_transform(tracks['track', 'genre_top'])
labels_onehot = pd.DataFrame(labels_onehot, index=tracks.index)

Load audio samples in parallel using `multiprocessing` so as to maximize CPU usage when decoding MP3s and making some optional pre-processing. There are multiple ways to load a waveform from a compressed MP3:
* librosa uses audioread in the backend which can use many native libraries, e.g. ffmpeg
    * resampling is very slow --> use `kaiser_fast`
    * does not work with multi-processing, for keras `fit_generator()`
* pydub is a high-level interface for audio modification, uses ffmpeg to load
    * store a temporary `.wav`
* directly pipe ffmpeg output
    * fastest method
* [pyAV](https://github.com/mikeboers/PyAV) may be a fastest alternative by linking to ffmpeg libraries

In [28]:
# Just be sure that everything is fine. Multiprocessing is tricky to debug.
utils.FfmpegLoader().load(utils.get_audio_path(AUDIO_DIR, 2))
SampleLoader = utils.build_sample_loader(AUDIO_DIR, labels_onehot, utils.FfmpegLoader())
SampleLoader(train, batch_size=2).__next__()[0].shape

(2, 1321967)

In [29]:

# Keras parameters.
NB_WORKER = len(os.sched_getaffinity(0))  # number of usable CPUs
params = {}

def as_gen(sample_loader):
    """Wrap a SampleLoader iterator as a true Python generator for Keras 3 compatibility."""
    while True:
        yield next(sample_loader)


### 2.1 Fully connected neural network

* Two layers with 10 hiddens is no better than random, ~11%.

Optimize data loading to be CPU / GPU bound, not IO bound. Larger batches means reduced training time, so increase batch time until memory exhaustion. Number of workers and queue size have no influence on speed.

In [30]:

loader = utils.FfmpegLoader(sampling_rate=2000)
SampleLoader = utils.build_sample_loader(AUDIO_DIR, labels_onehot, loader)
print('Dimensionality: {}'.format(loader.shape))

keras.backend.clear_session()

with tf.device(DEVICE):
    model = keras.models.Sequential()
    model.add(keras.Input(shape=loader.shape))
    model.add(Dense(1000))
    model.add(Activation("relu"))
    model.add(Dense(100))
    model.add(Activation("relu"))
    model.add(Dense(labels_onehot.shape[1]))
    model.add(Activation("softmax"))

    optimizer = keras.optimizers.SGD(learning_rate=0.1, momentum=0.9, nesterov=True)
    model.compile(optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# model.fit(as_gen(SampleLoader(train, batch_size=64)), steps_per_epoch=len(train)//64, epochs=2)
# loss = model.evaluate(as_gen(SampleLoader(val, batch_size=64)), steps=len(val)//64)
# loss = model.evaluate(as_gen(SampleLoader(test, batch_size=64)), steps=len(test)//64)

# loss


Dimensionality: (59953,)


### 2.2 Convolutional neural network

* Architecture: [End-to-end learning for music audio](http://www.mirlab.org/conference_papers/International_Conference/ICASSP%202014/papers/p7014-dieleman.pdf), Sander Dieleman, Benjamin Schrauwen.
* Missing: track segmentation and class averaging (majority voting)
* Compared with log-scaled mel-spectrograms instead of strided convolution as first layer.
* Larger net: http://benanne.github.io/2014/08/05/spotify-cnns.html

In [31]:

loader = utils.FfmpegLoader(sampling_rate=16000)
#loader = utils.LibrosaLoader(sampling_rate=16000)
SampleLoader = utils.build_sample_loader(AUDIO_DIR, labels_onehot, loader)

keras.backend.clear_session()

with tf.device(DEVICE):
    model = keras.models.Sequential()
    model.add(keras.Input(shape=loader.shape))
    model.add(Reshape((-1, 1)))
    print(model.output_shape)

    model.add(Conv1D(128, 512, strides=512))
    print(model.output_shape)
    model.add(Activation("relu"))

    model.add(Conv1D(32, 8))
    print(model.output_shape)
    model.add(Activation("relu"))
    model.add(MaxPooling1D(4))

    model.add(Conv1D(32, 8))
    print(model.output_shape)
    model.add(Activation("relu"))
    model.add(MaxPooling1D(4))

    print(model.output_shape)
    #model.add(Dropout(0.25))
    model.add(Flatten())
    print(model.output_shape)
    model.add(Dense(100))
    model.add(Activation("relu"))
    print(model.output_shape)
    model.add(Dense(labels_onehot.shape[1]))
    model.add(Activation("softmax"))
    print(model.output_shape)

    optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
    #optimizer = keras.optimizers.Adam()
    model.compile(optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# model.fit(as_gen(SampleLoader(train, batch_size=10)), steps_per_epoch=len(train)//10, epochs=20)
# loss = model.evaluate(as_gen(SampleLoader(val, batch_size=10)), steps=len(val)//10)
# loss = model.evaluate(as_gen(SampleLoader(test, batch_size=10)), steps=len(test)//10)

# loss


(None, 479625, 1)
(None, 936, 128)
(None, 929, 32)
(None, 225, 32)
(None, 56, 32)
(None, 1792)
(None, 100)
(None, 8)


### 2.3 Recurrent neural network

## 3 Deep learning on extracted audio features

Look at:
* Pre-processing in Keras: https://github.com/keunwoochoi/kapre
* Convolutional Recurrent Neural Networks for Music Classification: https://github.com/keunwoochoi/icassp_2017
* Music Auto-Tagger: https://github.com/keunwoochoi/music-auto_tagging-keras
* Pre-processor: https://github.com/bmcfee/pumpp

### 3.1 ConvNet on MFCC

* Architecture: [Automatic Musical Pattern Feature Extraction Using Convolutional Neural Network](http://www.iaeng.org/publication/IMECS2010/IMECS2010_pp546-550.pdf), Tom LH. Li, Antoni B. Chan and Andy HW. Chun
* Missing: track segmentation and majority voting.
* Best seen: 17.6%

In [32]:
class MfccLoader(utils.Loader):
    raw_loader = utils.FfmpegLoader(sampling_rate=22050)
    #shape = (13, 190)  # For segmented tracks.
    shape = (13, 2582)
    def load(self, filename):
        import librosa
        x = self.raw_loader.load(filename).astype(np.float32)
        # Each MFCC frame spans 23ms on the audio signal with 50% overlap with the adjacent frames.
        mfcc = librosa.feature.mfcc(y=x, sr=22050, n_mfcc=13, n_fft=512, hop_length=256)
        return mfcc

loader = MfccLoader()
SampleLoader = utils.build_sample_loader(AUDIO_DIR, labels_onehot, loader)
loader.load(utils.get_audio_path(AUDIO_DIR, 2))[0].shape

(2582,)

In [33]:

keras.backend.clear_session()

with tf.device(DEVICE):
    model = keras.models.Sequential()
    model.add(keras.Input(shape=loader.shape))
    model.add(Reshape((*loader.shape, 1)))
    print(model.output_shape)

    model.add(Conv2D(3, (13, 10), strides=(1, 4)))
    model.add(Activation("relu"))
    print(model.output_shape)

    model.add(Conv2D(15, (1, 10), strides=(1, 4)))
    model.add(Activation("relu"))
    print(model.output_shape)

    model.add(Conv2D(65, (1, 10), strides=(1, 4)))
    model.add(Activation("relu"))
    print(model.output_shape)

    model.add(Flatten())
    print(model.output_shape)
    model.add(Dense(labels_onehot.shape[1]))
    model.add(Activation("softmax"))
    print(model.output_shape)

    optimizer = keras.optimizers.SGD(learning_rate=1e-3)
    #optimizer = keras.optimizers.Adam()
    model.compile(optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    model.fit(as_gen(SampleLoader(train, batch_size=16)), steps_per_epoch=len(train)//16, epochs=3)
    loss = model.evaluate(as_gen(SampleLoader(val, batch_size=16)), steps=len(val)//16)
    loss = model.evaluate(as_gen(SampleLoader(test, batch_size=16)), steps=len(test)//16)

    loss


(None, 13, 2582, 1)
(None, 1, 644, 3)
(None, 1, 159, 15)
(None, 1, 38, 65)
(None, 2470)
(None, 8)
Epoch 1/3
  1/400 [..............................] - ETA: 2:49 - loss: 8.8113 - accuracy: 0.1875
Ignoring /mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_small/108/108925.mp3 (error: Command '['ffmpeg', '-i', '/mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_small/108/108925.mp3', '-f', 's16le', '-acodec', 'pcm_s16le', '-ac', '1', '-ar', '22050', '-']' returned non-zero exit status 234.).
 37/400 [=>............................] - ETA: 23:17 - loss: 2.2611 - accuracy: 0.1201
Ignoring /mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_small/099/099134.mp3 (error: Command '['ffmpeg', '-i', '/mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_small/099/099134.mp3', '-f', 's16le', '-acodec', 'pcm_s16le', '-ac', '1', '-ar', '22050', '-']' returned non-zero exit status 234.).

Ignoring /mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/

KeyboardInterrupt: 